# Bits and Bytes in C

*Run a cell with ▶ (or Shift+Enter): Sphinx compiles and runs it on your computer. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Read, set, clear and flip bits with &, |, ^, ~ and shifts, on unsigned fixed-width types such as uint32_t.
- Pack several small fields into one integer and unpack them again with masks and shifts.
- Look at the bytes and bits of any value, such as a float, by copying it into an integer with memcpy.

**Exercises** in this notebook end with `check(...)` lines: minimal tests that print ✓ when your code is right. Run the next cell once first, so they can use `check`.

In [ ]:
// shared: check_int and check_str print ✓ or ✗; the exercises below use them
void check_int(const char *what, long actual, long expected) {
    if (actual == expected) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: expected %ld, got %ld\n", what, expected, actual);
    }
}

void check_str(const char *what, const char *actual, const char *expected) {
    if (actual == NULL) {
        actual = "(NULL)";
    }
    if (strcmp(actual, expected) == 0) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: expected \"%s\", got \"%s\"\n", what, expected, actual);
    }
}

## In short

C works directly on the bits of integers: `&` (and), `|` (or), `^` (xor), `~` (not), `<<` and `>>` (shifts). Use **unsigned** fixed-width types from `<stdint.h>`, such as `uint32_t`: their size is certain, and their shifts are fully defined.

```c
#include <stdio.h>
#include <stdint.h>
#include <string.h>

int main(void) {
    uint32_t flags = 0;
    flags |= 1u << 3;                     // set bit 3
    flags ^= 1u << 0;                     // flip bit 0
    int bit3 = (flags >> 3) & 1;          // read bit 3
    flags &= ~(1u << 0);                  // clear bit 0
    printf("%u %d\n", (unsigned) flags, bit3);   // 8 1

float f = 1.0f;
    uint32_t bits;
    memcpy(&bits, &f, sizeof bits);       // the float's own bits
    printf("0x%08X\n", (unsigned) bits); // 0x3F800000
    return 0;
}
```

A **mask** selects a field: `(value >> 5) & 0x3F` takes the 6 bits that start at bit 5.

#### 🎞️ Reading a bit field with a shift and a mask

![To read bits 5 to 10 of 0xABCD, shift right by 5 to bring them to the bottom, then AND with 0x3F to keep 6 bits: 011110, or 30](shift-mask-steps.svg)

**Shifts, masks and wrap-around:** change `value` and `start` to read other fields.

In [ ]:
uint16_t value = 0xABCD;
int start = 5;
unsigned field = (value >> start) & 0x3F;         /* 6 bits from bit 5 */
printf("0x%04X: bits %d..%d are %u (0x%02X)\n", (unsigned) value, start, start + 5, field, field);
for (int k = 0; k < 8; k++) {
    printf("1u << %d = %3u\n", k, 1u << k);
}
uint8_t byte = 255;
byte = (uint8_t) (byte + 1);                      /* wraps around to 0 */
printf("255 + 1 in a uint8_t: %u\n", (unsigned) byte);

### ✏️ Exercise: Turn a bit on

Return x with bit k set to 1, using `|` and a shifted 1. A bit that is already on stays on.

In [ ]:
unsigned set_bit(unsigned x, int k) {
    // TODO: OR x with 1u << k
    return x;
}

int main(void) {
    check_int("bit 3 of 0", set_bit(0, 3), 8);
    check_int("already on", set_bit(8, 3), 8);
    check_int("bit 4 of 1", set_bit(1, 4), 17);
    return 0;
}

<details>
<summary>Show an answer</summary>

```c
unsigned set_bit(unsigned x, int k) {
    return x | (1u << k);
}

int main(void) {
    check_int("bit 3 of 0", set_bit(0, 3), 8);
    check_int("already on", set_bit(8, 3), 8);
    check_int("bit 4 of 1", set_bit(1, 4), 17);
    return 0;
}
```

</details>

**Watch out:** shift only unsigned values, by less than their width: other shifts are undefined behavior or depend on the compiler.

### ✏️ Exercise: The low nibble

Return the lowest 4 bits of x (one hex digit) with a mask.

In [ ]:
unsigned low_nibble(unsigned x) {
    // TODO: AND x with 0xF
    return 0;
}

int main(void) {
    check_int("0xAB", low_nibble(0xAB), 0xB);
    check_int("0x30", low_nibble(0x30), 0);
    check_int("0xFFFF", low_nibble(0xFFFF), 15);
    return 0;
}

<details>
<summary>Show an answer</summary>

```c
unsigned low_nibble(unsigned x) {
    return x & 0xFu;
}

int main(void) {
    check_int("0xAB", low_nibble(0xAB), 0xB);
    check_int("0x30", low_nibble(0x30), 0);
    check_int("0xFFFF", low_nibble(0xFFFF), 15);
    return 0;
}
```

</details>

## Readings

- [Bitwise Operations](https://beej.us/guide/bgc/html/split/bitwise-operations.html) (Beej's Guide to C Programming, chapter 24, 4 min): AND, OR, XOR and NOT, then shifts, and the warning about shifting signed values. Short: the operators are the ones you used in CS Fundamentals, now in C.
- [Fixed Width Integer Types](https://beej.us/guide/bgc/html/split/fixed-width-integer-types.html) (Beej's Guide to C Programming, chapter 37, 5 min): Optional: uint8_t, uint32_t and friends, their limits, and the PRIu32-style format specifiers. Sphinx's compilers also print uint32_t with %u, which the challenges use.

## Practice

The notebook covered the basics. Now apply them in the unit's challenges, in the Sphinx sidebar, where Submit runs every test, hidden ones included:

- **Count the Bits** · Easy
- **RGB565 Colors** · Easy
- **Float Anatomy** · Medium
- **CRC-32** · Hard